# GuardRails

In [6]:
from dotenv import load_dotenv
load_dotenv()

import os
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

## Section 1: Two Approaches to Guardrails

**Deterministic Guardrails**

These are rule-based checks: regex patterns, keyword matching, explicit logic. They are fast, predictable, and cost-effective -- but they can miss nuanced violations.

In [7]:
import re

def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked."""
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]

print("=== Deterministic Guardrail Demo ===")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "BLOCKED" if blocked else "ALLOWED"
    print(f"{status}: {inp}")

=== Deterministic Guardrail Demo ===
BLOCKED: How do I hack into a database?
ALLOWED: What is the capital of France?
BLOCKED: Explain how malware spreads


Fast and cheap -- but notice it would also block 
“Explain how companies protect against malware,”
 which is a perfectly legitimate question. Keyword matching has no understanding of intent.

**Model-Based Guardrails**

These use an LLM or classifier for semantic understanding. They catch subtle and nuanced issues that keyword matching misses -- but they are slower and more expensive.

In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI

def model_based_guardrail(text: str) -> str:
    """Uses an LLM to evaluate content safety. Returns SAFE or UNSAFE."""
    model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
    prompt = f"""Is the following user input safe to process?
Reply with only 'SAFE' or 'UNSAFE'.

Input: {text}"""
    result = model.invoke([{"role": "user", "content": prompt}])
    return result.content.strip()

print("=== Model-Based Guardrail Demo ===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "UNSAFE" if "UNSAFE" in verdict else "SAFE"
    print(f"{status}: {inp}")

=== Model-Based Guardrail Demo ===
UNSAFE: How do I hack into a database?
SAFE: What is the capital of France?
SAFE: Explain how malware spreads


## Section 2: Built-in PII Detection Middleware

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_google_genai  import ChatGoogleGenerativeAI
from langchain_core.tools import tool

In [10]:
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"

In [11]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from dotenv import load_dotenv
load_dotenv()
# Set Gemini API key (recommended: store in environment variable)
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

# Define the model
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

# Create agent with PII Middleware
agent = create_agent(
    model=model,
    tools=[customer_lookup],
    middleware=[
        # Redact emails before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),

        # Mask credit cards
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),

        # Block API keys
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")

Agent with PII middleware created successfully!


In [12]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "My email is john.doe@example.com and my card is "
            "5105-1051-0510-5100. Can you help me?"
        )
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
[{'type': 'text', 'text': 'I found a customer record for you. How can I help you today?', 'extras': {'signature': 'CrICAb4+9vsb5ElLPSKzgVEijW7uOszQqi3X/5V7DPv2aw2N2Uf6dDkmYc+Y6zqzZ7XYkbjlGf/3GrBXdNnahcdkL194t4kEGzYyaQNUhMCPVWBJVqjlCbSGJho1SSen2gQelEjwCzmQR69hieJPr4JaWJQEyAHLZ5j+8PisIweXSWxgXSqGCwQnmv0AIfYCYSdsqu2rGHkDd9nmsZUzRw+Z5oMtThVkLZDhm/LC/zn8QzNs3Mg8Dcw1Jv8vKO73uG8vELlthL64TlKBnHvOIlqiNUQKKfK7vBsTSa8I8eajfVIupotXVeuagc1L9kpm+GgoLidLwQKOZNdhelBh3OiypFXU5iOoPTTaffpzUUjCct+ixhPFGLhjdLAy0IQmA9nPCFwLkMM+JKcejt1FQYh9le16'}}]


In [13]:
result["messages"][-1].content

[{'type': 'text',
  'text': 'I found a customer record for you. How can I help you today?',
  'extras': {'signature': 'CrICAb4+9vsb5ElLPSKzgVEijW7uOszQqi3X/5V7DPv2aw2N2Uf6dDkmYc+Y6zqzZ7XYkbjlGf/3GrBXdNnahcdkL194t4kEGzYyaQNUhMCPVWBJVqjlCbSGJho1SSen2gQelEjwCzmQR69hieJPr4JaWJQEyAHLZ5j+8PisIweXSWxgXSqGCwQnmv0AIfYCYSdsqu2rGHkDd9nmsZUzRw+Z5oMtThVkLZDhm/LC/zn8QzNs3Mg8Dcw1Jv8vKO73uG8vELlthL64TlKBnHvOIlqiNUQKKfK7vBsTSa8I8eajfVIupotXVeuagc1L9kpm+GgoLidLwQKOZNdhelBh3OiypFXU5iOoPTTaffpzUUjCct+ixhPFGLhjdLAy0IQmA9nPCFwLkMM+JKcejt1FQYh9le16'}}]

In [14]:
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
except Exception as e:
    print(f"Blocked as expected: {e}")

Blocked as expected: Detected 1 instance(s) of api_key in text content


## Section 3: Built-in Human-in-the-Loop Middleware

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

In [16]:
@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

In [17]:
@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

In [18]:
# Define the model
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

# Create agent with HITL middleware
hitl_agent = create_agent(
    model=model,
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,       # Require approval
                "delete_records": True,    # Require approval
                "search_web": False,       # Auto-approve
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)

In [19]:
# Step 1: Invoke -- agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the project report that project is Done and the report is ready?" }]},
    config=config
)

print("=== Agent paused -- awaiting human approval ===")

=== Agent paused -- awaiting human approval ===


In [20]:
result["messages"][-1]

AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email', 'arguments': '{"body": "The project is done and the report is ready.", "to": "team@company.com", "subject": "Project Report - Project Done"}'}, '__gemini_function_call_thought_signatures__': {'e69db93b-6b0b-4838-baf5-78b46b7a0252': 'CpAEAb4+9vv/jnUckb3nzY33t3UxLx4Ty1p9i133xwl3fZ9Mezf4vxj/5jsTuCj6inHc+7XS+9bj+jUlQnyx6AUPe+wIuUu4bqNg/beY26S5kD6ngu1JOhRPK+0y6wh0FY7uc1Due1KxZ/jxHs7t3HXVk215ASEmmGSzocqlJp13fi2LftM5eENTpD9Bopb9Q6isZ6PtvVKqEuKDU/Axf/tOnPJ00lbz3wNlvnttuCE1J2/1jdn3rfJwr2gmTvqIfsfmyIJPP665Wa7ufDPRlY5EvMOk+VE/ulIDDnQV9/G37KMGlWxCBVNGXYbKt9N0m8G+IsnfusqAAncPBQkUGSo+9SPWv8jZGvmKRbnR9qpmWQF+kqmpMCsaNHFgTpOhGsrlJDGPGJ+HGcFKa22SGjIXJKP0cOe4TQQq0NADLcNp8XVjEQwWf4vXba9X3ILZd7vMJOuGiWC0vf9G+0ws4mLVoObr8EfMBh+qVzqs9An64wEsrXN8RPfAR3Fu6iydGc9cwysNYl5a0pKAFPbch/4w5zZ57sAUayQkVt2WH0EnUbWvWw5mqULpXThOAqWnpyzRsvMIUx3YxlUctNQvxKZhcJjn0JIrPplH6n0/yiBZMJwaX+wphpRIjydLYHYrzSMIEGXhSpM2RjTciBO0lFRjyj+SzsPYzEysyXBS+f4bZ9

In [21]:
# Step 2: Human reviews and APPROVES
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config  # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

KeyboardInterrupt: 

In [22]:
approved_result["messages"]

NameError: name 'approved_result' is not defined

**rejection**

In [ ]:
# Alternative -- Human REJECTS
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2
)

rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

=== Rejected! Final response ===
I'm sorry, I wasn't able to delete the records. It seems that the tool call was rejected.


## Section 4: Custom Before-Agent Guardrail (Input Filter)

In [23]:
from typing import Any
from langchain.agents.middleware import (
    AgentMiddleware, AgentState, hook_config
)
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

In [24]:
class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything --
    zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"Blocked -- keyword detected: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing "
                            "inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
        return None

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

# Create agent with content filter
model = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")
filtered_agent = create_agent(
    model=model,
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=[
                "hack", "exploit", "malware", "jailbreak", "bypass"
            ]
        ),
    ],
)

In [ ]:
# Test 1: Safe request -- should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("Safe request response:")
print(result["messages"][-1].content)

Safe request response:
[{'type': 'text', 'text': "I'm sorry, I couldn't find a definition for machine learning using my search tool.", 'extras': {'signature': 'CoYEAb4+9vvRY7tuNWcDZcP/HNx+vEnKktXMB41PpICobM6xhqDWoZCdtIGSwpTQS1hJULGreorXMArUz/LZ7E6NfcKuDAdluhIA1chiGIzBHTwkBunpTnaq2q38M0OFMm4wv3jdJjOMdoUnMYHTBT4qbh7FZjWvsPXWcA38j6jEjFiTTUYk2r8vy5j1K8otICWjGTo4eJlVGPb0YTXZdzIrkg90QtrAs2/viV1LFCdkeaWgCuuPgokJAl3ck8ZzHtISWEqbSvxVatwXQi2eKp4vEK02mWUPqfC3On/bZHeFbrF4xuwL4rJnAcvpGEnYEbdFP2fDDLKBf9DTnzFFVSWRxTnst+UilzIaCYJbmK2b6bra23tG5b/M0ZIhjeBwoiZjoDWLHJ4RdM64XOkWCLMCSyaNjvtBzK6fTT/Afbgwxkm5x8sGtv28braneGHVlERpcGQnnXJvDRYbzpEObOFp/WAykpDIhUH7qbm22yhA0Nty2GP87MX11YbetYWNg7CVbvx8puS5+ZEuEf52hi/7jc2tl30MAAkgjFc0ym6WYBBD+FEHR1zO1hkvEidc4geta7qvvkfvMnLHmUGpTUnYq+U7H+LNnDjC6Nx8cHcmifVvrCPp9xBnpaEzGf01q6QjbIYudo7KJAMLv5DxCcWdd219ovg9WOPu/W/+5xC/3gA1X2m9ORQ='}}]


In [ ]:
for x in result["messages"]:
    print(x)

content='What is machine learning?' additional_kwargs={} response_metadata={} id='d90e120d-3caa-4907-81e1-9af1811e41a8'
content='' additional_kwargs={'function_call': {'name': 'search_tool', 'arguments': '{"query": "What is machine learning?"}'}, '__gemini_function_call_thought_signatures__': {'49781124-cef1-435f-9a3c-f9015307a681': 'Cq4CAb4+9vv0fh7Nuk3Bxy2OmeiIgK3g6Zr11E4X4ZL5hPK1ATJfd1y+9fy0zCplmbT2isg/RGXTIqcJs1caaQjyZRVr4E1QRcKjdmkCg3qjf47rFnJenRxtomx4k/emPk7dghBrKV0jLx3FWeV0rg+vZK8vY0usR4pJOyPFcyG0EH63TcfPIkZDY1WvtV1/yHlKYfTa3Cu4DwXTdRtCFNVpc55umCDHBzLPRUAus4ZVnfwYi4UuKi8c2IADD89J0cgaq2Tfi9snV2VtZoT179Kfq4Tf3SR6mQH3Idl8bJ+ZzKUeCSllZ3IjT3X1JD0WiShZv8pe4aY5ocNmYPjMGFWQuwk2wrfQx620644VgR5tr68mUTF5hKXTmsyFDuTyWQey/NfFFa4bw1WZ4dTgQGo='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019cd09b-e138-71a0-af9f-14294ca09392-0' tool_calls=[{'name': 'search_tool', 'args': {'query': 'What is mac

In [ ]:
# Test 2: Unsafe request -- should be blocked
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})
print("Unsafe request response:")
print(result["messages"][-1].content)

Blocked -- keyword detected: 'hack'
Unsafe request response:
I cannot process requests containing inappropriate content. Please rephrase your request.


## Section 5: Custom After-Agent Guardrail (Output Safety)

In [25]:
from typing import Any
from langchain.agents.middleware import (
    AgentMiddleware, AgentState, hook_config
)
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool

In [39]:
class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        # Use a smaller, cheaper model for the safety check
        self.safety_model = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")

    @hook_config(can_jump_to=["end"])
    def after_agent(   self, state: AgentState, runtime: Runtime ) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a lightweight model as the safety judge
        safety_prompt = f"""Evaluate if this AI response is safe \
and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}"""

        result = self.safety_model.invoke(
            [{"role": "user", "content": safety_prompt}]
        )
        print("result-----"  , result)
        if "UNSAFE" in result.content.upper():
            print("Output flagged as UNSAFE -- replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None


@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"

model = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")
safe_agent = create_agent(
    model=model,
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

In [40]:
# Test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "how to hack generte answer?"}]
})
print("Response:")
print(result["messages"][-1].content)

result----- content='SAFE' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019cd0ae-45f8-7180-a8cf-0983f273a72e-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 563, 'output_tokens': 79, 'total_tokens': 642, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 78}}
Response:
[{'type': 'text', 'text': 'I cannot assist with requests that involve hacking or any other illegal activities. My purpose is to be helpful and harmless, and that includes promoting ethical and legal behavior.', 'extras': {'signature': 'Cv0DAb4+9vv/7U14O/oz0kwXy+FKryHxIttoq0dV8Y7P1qSAcs1cc1w4Qad87z6kTZr5JrlAjMF1naXbxHPZXdXnkF6hDIyQnrQrRRqxN3TMyZZWlHiNul5jO4JTiXdXZfsdOOMjRnElXvnnabJGz3EwMDU5wAFqB35TblWZQvRZ337Yh4PpO4bJRD1SwBOVl+Zw9UZW0Bk9TGilpoY/aUFd5h/dt2yGr/pc9tBwkhohn6Ow/GGZUMrVuXwOLLxeN17mP0LL549wObWGCQCIH9s9kbnEPxgytcPOtWedFnNPQGvnSMgDjGwQaHWF1fY0

In [37]:
result

{'messages': [HumanMessage(content='how to hack ', additional_kwargs={}, response_metadata={}, id='729cfc2a-84cd-4837-87ed-f876f33bea53'),
  AIMessage(content=[{'type': 'text', 'text': 'I cannot assist with that request. Hacking can have serious legal consequences and can cause significant harm. My purpose is to be helpful and harmless, and that includes not providing information that could be used for illegal activities.', 'extras': {'signature': 'CrIEAb4+9vvqYjGTdrtjpao00E1yS5vbJVNYdXicCAIqYnC7a+MHnjy/Hw6oGaRjrPFS0QJGhUAWeKpZGpTVj6aaEKW7Jmq6+MXcztjV6iTFvaADVYU1DHfPpKUHMThbeyVsOE/W8bdo+yZDTv9eVfHoDpksUrQ9tA+gZoiwAayi+wK0qYqxi3XGcMU0qPB7LAbJRM8dokltXdOT/P+AyfbOk62nZMAOdYPLk2rLXyEy+xfo22eOia6biZWVZjLYPHJFyd9BzIi8+dS1JNRblu1tu7Qyw3CeuAQg+r23RA8XbJkKIQwSw6r5vD+6+6uNGLawB7SnUzbsmbixr/VDFzdn3P5hv7PIzcRtyLmRF3CPQ4soIEAreNzi9PWmS4uGzcCDZJCWm6VAvG016BR2oKXJERr2xeKYja+k6vaSn4Y5Ew87bRoGmxjMMlj1ljUcYXSthHg/7fbTvrP87ymNozGJ6tP/tAEbuAd1LIn++U3z0WGpR2svaA+rbUseNHYWifozKk2vLwhWjrttJq/J1snXAiNZXL/4X0N

## Section 6: Layered / Combined Guardrails

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import (
    PIIMiddleware, HumanInTheLoopMiddleware
)
import os
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}"

# Full layered guardrail stack
model = ChatGoogleGenerativeAI(model = "gemini-2.5-flash" , api_key=os.getenv("GOOGLE_API_KEY"))
production_agent = create_agent(
    model=model,
    tools=[search_tool, send_email_tool],
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware"]
        ),

        # Layer 2: PII redaction on input
        PIIMiddleware(
            "credit_card", strategy="mask", apply_to_input=True
        ),

        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": True,
                "search_tool": False,
            }
        ),

        # Layer 4: PII redaction on output
        PIIMiddleware(
            "email", strategy="redact", apply_to_output=True
        ),

        # Layer 5: Model-based output safety
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)


print("Production-grade agent with 5-layer guardrails created!")

Production-grade agent with 5-layer guardrails created!


In [33]:
config = {"configurable": {"thread_id": "session_001"}}

result = production_agent.invoke(
    {"messages": [{"role": "user", "content": "How to hack?"}]},
    config=config
)

Blocked -- keyword detected: 'hack'


In [34]:
result

{'messages': [HumanMessage(content='How to hack?', additional_kwargs={}, response_metadata={}, id='fb62a7b8-5c5a-4b00-9f37-526407409293'),
  AIMessage(content='I cannot process requests containing inappropriate content. Please rephrase your request.', additional_kwargs={}, response_metadata={}, id='88b98386-14fe-4852-817f-2755fef11b2c', tool_calls=[], invalid_tool_calls=[])]}